In [0]:
import os

# Define the exact path from your catalog explorer screen
huldra_path = "/Volumes/equinor_asa_p_id_and_scd_of_huldra/public/huldra_pid_scd"

# List out the actual engineering files inside
files = os.listdir(huldra_path)
print(f"Total files found: {len(files)}")
for file in files:
    print(f"- {file}")

In [0]:
import os

huldra_path = "/Volumes/equinor_asa_p_id_and_scd_of_huldra/public/huldra_pid_scd"

# Define the paths to the subfolders
pid_path = os.path.join(huldra_path, "pid")
scd_path = os.path.join(huldra_path, "scd")

# Function to peek inside a directory safely
def peek_directory(dir_path, name):
    if os.path.exists(dir_path):
        files = os.listdir(dir_path)
        print(f"\n📁 Inside the '{name}' directory (Found {len(files)} files):")
        # Print the first 10 files as a sample so we don't overwhelm the screen
        for file in files[:10]:
            print(f"  └── {file}")
        if len(files) > 10:
            print("  └── ... and more files.")
    else:
        print(f"[WARNING] {name} directory not found at {dir_path}")

# Run the exploration
peek_directory(pid_path, "P&ID Drawings")
peek_directory(scd_path, "System Control Diagrams (SCD)")

In [0]:
import zipfile
import os

# Define source zip paths from your previous output
huldra_path = "/Volumes/equinor_asa_p_id_and_scd_of_huldra/public/huldra_pid_scd"
pid_zip_path = os.path.join(huldra_path, "pid", "HuldraPIDs.zip")

# Define a local directory in Databricks to extract the files into
extracted_target_dir = "/tmp/huldra_extracted_pids"
os.makedirs(extracted_target_dir, exist_ok=True)

print("[INFO] Extracting real Huldra P&ID drawings to local workspace...")

# Unzip the file
with zipfile.ZipFile(pid_zip_path, 'r') as zip_ref:
    zip_ref.extractall(extracted_target_dir)

print("[SUCCESS] Extraction complete!")

# List the actual engineering files now available
extracted_files = os.listdir(extracted_target_dir)
print(f"\n📁 Total extracted drawings found: {len(extracted_files)}")
print("Listing the first 15 drawings:")
for file in sorted(extracted_files)[:15]:
    print(f"  └── {file}")

In [0]:
import os

base_extracted_dir = "/tmp/huldra_extracted_pids"

# Paths to the nested subfolders revealed by your last run
folder_1 = os.path.join(base_extracted_dir, "HuldraPIDs")
folder_2 = os.path.join(base_extracted_dir, "Semantum Huldra P&IDs")

def inspect_subfolder(folder_path, folder_name):
    if os.path.exists(folder_path):
        # Check if it's a directory
        if os.path.isdir(folder_path):
            contents = os.listdir(folder_path)
            print(f"\n📁 Contents of '{folder_name}' (Found {len(contents)} items):")
            # Show the first 15 files to keep the screen clean
            for item in sorted(contents)[:15]:
                print(f"  └── {item}")
            if len(contents) > 15:
                print("  └── ... and more files.")
        else:
            print(f"[INFO] '{folder_name}' is a file: {folder_path}")
    else:
        print(f"[ERROR] Path not found: {folder_path}")

# Run inspection on both nested directories
inspect_subfolder(folder_1, "HuldraPIDs")
inspect_subfolder(folder_2, "Semantum Huldra P&IDs")

In [0]:
%pip install pystac-client planetary-computer

In [0]:
# DIAGNOSTIC CELL — run this alone before anything else
import requests

url = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
params = {
    "$filter": (
        "Collection/Name eq 'SENTINEL-5P' and "
        "Attributes/OData.CSC.StringAttribute/any("
        "att:att/Name eq 'productType' and "
        "att/OData.CSC.StringAttribute/Value eq 'L2__CH4___')"
    ),
    "$orderby": "ContentDate/Start desc",
    "$top": 1
}

resp = requests.get(url, params=params, timeout=15)
product = resp.json()["value"][0]
print(list(product.keys()))
print()
for k, v in product.items():
    print(f"{k}: {v}")

In [0]:
dbutils.library.restartPython()

In [0]:
# Clear any old widgets and create clean new dropdown inputs
dbutils.widgets.removeAll()

# Dropdown for selecting which asset sensor is failing
dbutils.widgets.dropdown("Target_Sensor", "PT-HB20-01", ["PT-HB20-01", "PT-HA24-01", "PT-HO45-01"])

# Input for manually adjusting live system torque to test the physics engine
dbutils.widgets.text("Live_Torque_Nm", "80")

print("[UI] Interactive engineering control panel initialised at the top of the workspace.")

In [0]:
"""
Project Sentinel — Intelligent Multi-Source Safety Dashboard
Microsoft Agents League Hackathon 2026

Data sources (all public):
  - UCI AI4I 2020 Predictive Maintenance Dataset (CC BY 4.0)
  - ESA Sentinel-5P TROPOMI via Microsoft Planetary Computer (open access)
  - Equinor Huldra Open P&ID Data (published by Equinor on Databricks Marketplace)

NOTE: This prototype uses only publicly available data.
      In a production deployment, client SCADA feeds and document stores
      would be connected via Azure IoT Hub and Azure AI Search respectively.
"""

# ── INSTALL (run once in Databricks) ─────────────────────────────────────────
# %pip install -q gradio pillow pymupdf openai pystac-client planetary-computer

import os, json, math, io, warnings
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr
from PIL import Image

warnings.filterwarnings("ignore", category=FutureWarning)

# ── AZURE OPENAI CONFIG ───────────────────────────────────────────────────────
# Set these as Databricks secrets or environment variables — never hard-code keys
AZURE_OPENAI_ENDPOINT = os.environ.get("AZURE_OPENAI_ENDPOINT", "")   # e.g. https://your-hub.openai.azure.com/
AZURE_OPENAI_KEY      = os.environ.get("AZURE_OPENAI_KEY", "")
AZURE_DEPLOYMENT      = os.environ.get("AZURE_DEPLOYMENT", "gpt-4o")  # your Foundry deployment name

def _get_openai_client():
    """Returns an AzureOpenAI client, or None if credentials are not configured."""
    if not AZURE_OPENAI_ENDPOINT or not AZURE_OPENAI_KEY:
        return None
    try:
        from openai import AzureOpenAI
        return AzureOpenAI(
            azure_endpoint=AZURE_OPENAI_ENDPOINT,
            api_key=AZURE_OPENAI_KEY,
            api_version="2024-02-01"
        )
    except ImportError:
        return None


# ── DATA LAYER ────────────────────────────────────────────────────────────────

def fetch_uci_telemetry(row_index: int) -> dict:
    """
    Downloads the UCI AI4I 2020 Predictive Maintenance dataset (CC BY 4.0)
    and returns a single row as a dict.
    Falls back to a realistic synthetic record on network failure.
    """
    try:
        url = "https://archive.ics.uci.edu/static/public/601/ai4i+2020+predictive+maintenance+dataset.zip"
        df = pd.read_csv(url, compression="zip")
        df.columns = [c.split("[")[0].strip().replace(" ", "_") for c in df.columns]
        row = df.iloc[max(0, min(int(row_index), 9999))]
        return {
            "rpm":             float(row["Rotational_speed"]),
            "torque":          float(row["Torque"]),
            "tool_wear":       int(row["Tool_wear"]),
            "air_temp":        float(row["Air_temperature"]),
            "process_failure": int(row["Machine_failure"]),
            "source":          "UCI Live"
        }
    except Exception:
        return {
            "rpm": 1350.0, "torque": 78.5, "tool_wear": 120,
            "air_temp": 298.1, "process_failure": 1, "source": "Fallback"
        }


def fetch_satellite_methane(bbox: list, date_range: str) -> dict:
    """
    Fetches the most recent Sentinel-5P L2 CH4 scene from
    ESA Copernicus Data Space (catalogue.dataspace.copernicus.eu).

    Switched from Planetary Computer which only holds SO2 data.
    The Copernicus OData API is free, no authentication needed for search,
    and confirmed working June 2026 (HTTP 200, live CH4 scenes returned).

    Note: the actual CH4 column value (methane_mixing_ratio_bias_corrected)
    lives inside the NetCDF file asset, not in catalogue metadata.
    We return scene ID and acquisition time to prove the feed is live.
    """
    import requests

    try:
        url = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
        params = {
            "$filter": (
                "Collection/Name eq 'SENTINEL-5P' and "
                "Attributes/OData.CSC.StringAttribute/any("
                "att:att/Name eq 'productType' and "
                "att/OData.CSC.StringAttribute/Value eq 'L2__CH4___')"
            ),
            "$orderby": "ContentDate/Start desc",
            "$top": 1
        }

        resp = requests.get(url, params=params, timeout=15)
        resp.raise_for_status()
        data = resp.json()

        if data.get("value"):
            p        = data["value"][0]

            # ContentDate is a nested dict: {"Start": "...", "End": "..."}
            acq_start = p["ContentDate"]["Start"][:16].replace("T", " ") + " UTC"
            acq_end   = p["ContentDate"]["End"][:16].replace("T", " ") + " UTC"

            # Online is a boolean True/False
            online    = "Yes" if p.get("Online") is True else "No"

            # ContentLength is in bytes
            size_mb   = round(p.get("ContentLength", 0) / 1_000_000, 2)

            # PublicationDate and OriginDate are flat strings
            pub_date  = p.get("PublicationDate", "")[:10]
            origin    = p.get("OriginDate", "")[:10]

            # S3Path gives the storage location
            s3_path   = p.get("S3Path", "N/A")

            return {
                "scene_id":     p["Name"],
                "acquired":     f"{acq_start} → {acq_end}",
                "published":    pub_date,
                "origin":       origin,
                "online":       online,
                "size_mb":      size_mb,
                "s3_path":      s3_path,
                "ch4_note":     "Column value inside NetCDF asset (not in catalogue metadata)",
                "product_type": "L2__CH4___ TROPOMI Methane",
                "status":       "LIVE — ESA Copernicus Data Space",
                "source":       "catalogue.dataspace.copernicus.eu"
            }

    except Exception:
        pass

    return {
        "scene_id":     "S5P_OFFL_L2__CH4____FALLBACK",
        "acquired":     "N/A",
        "published":    "N/A",
        "online":       "N/A",
        "size_mb":      0,
        "ch4_note":     "N/A (offline fallback)",
        "product_type": "L2__CH4___",
        "status":       "CACHED",
        "source":       "N/A"
    }


def count_pdf_references(file_path: str, system_code: str) -> int:
    """Counts occurrences of system_code in a plain-text PDF extraction."""
    if not os.path.exists(file_path):
        return 0
    try:
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            return max(1, f.read().count(system_code))
    except Exception:
        return 0


# ── PHYSICS / ENGINEERING CALCULATIONS ───────────────────────────────────────

def calc_fluid_transient_surge(torque: float) -> tuple:
    """
    Estimates worst-case pressure surge (Joukowski equation approximation)
    and minimum safe valve closure time to avoid fluid hammer.
    
    Reference: Joukowski, N. (1898). Über den hydraulischen Stoss in
    Wasserleitungsröhren. Mémoires de l'Académie Impériale des Sciences
    de St.-Pétersbourg.
    
    ⚠ Illustrative only — not validated for production use.
    """
    v_fluid   = torque / 10.0       # proxy: fluid velocity (m/s)
    a_wave    = 1200.0              # speed of sound in fluid (m/s), typical for crude oil
    rho       = 800.0               # density kg/m³
    surge_bar = round((rho * a_wave * v_fluid) / 1e5, 1)
    pipe_len  = 5.0                 # assumed pipe segment length (m)
    # Safe closure = at least 2L/a (Joukowski criterion)
    safe_time = round((2 * pipe_len / a_wave) + 2.0, 2) if surge_bar > 40.0 else 0.5
    return surge_bar, safe_time


# ── AZURE AI FOUNDRY REASONING ────────────────────────────────────────────────

def get_foundry_assessment(telemetry: dict, leak_rate: float,
                            surge_bar: float, system_code: str) -> str:
    """
    Calls Azure AI Foundry (GPT-4o) with structured telemetry context
    and returns a reasoned safety assessment.
    Falls back to a rule-based trace if Foundry is not configured.
    """
    client = _get_openai_client()

    if client:
        prompt = f"""You are an industrial safety analyst reviewing real-time asset telemetry.
Your role is to reason step-by-step (ReAct pattern) and recommend operator actions.
Never mandate automated shutdowns — always recommend field operator verification first.
Follow Goal Zero safety principles: no harm to people, assets, or environment.

Asset: {system_code}
Telemetry (UCI AI4I 2020 dataset, CC BY 4.0):
  RPM:            {telemetry['rpm']}
  Torque:         {telemetry['torque']} Nm
  Tool Wear:      {telemetry['tool_wear']} min
  Failure Flag:   {telemetry['process_failure']}

Derived indicators:
  Estimated fugitive emission proxy: {leak_rate} kg/hr (illustrative, OGMP 2.0 Level 4 methodology)
  Hydraulic transient surge:         {surge_bar} bar (Joukowski approximation)

Respond with:
THOUGHT: [your reasoning]
RISK LEVEL: [NOMINAL / WARNING / CRITICAL]
RECOMMENDED ACTION: [one clear sentence for the field operator]
CAVEAT: [one sentence on what should be verified by a qualified engineer]"""

        try:
            response = client.chat.completions.create(
                model=AZURE_DEPLOYMENT,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=300,
                temperature=0.2
            )
            return (
                f"🤖 [Azure AI Foundry — {AZURE_DEPLOYMENT}]\n\n"
                + response.choices[0].message.content
            )
        except Exception as e:
            return f"[Foundry call failed: {e}]\n\n" + _rule_based_trace(
                telemetry, leak_rate, surge_bar, system_code
            )
    else:
        # Foundry not configured — honest fallback
        return (
            "⚠️  Azure AI Foundry not configured (set AZURE_OPENAI_ENDPOINT + AZURE_OPENAI_KEY).\n"
            "Showing rule-based fallback trace:\n\n"
            + _rule_based_trace(telemetry, leak_rate, surge_bar, system_code)
        )


def _rule_based_trace(telemetry, leak_rate, surge_bar, system_code):
    """Deterministic fallback reasoning trace (no LLM required)."""
    if telemetry["process_failure"] == 1 or leak_rate > 8.0:
        return (
            f"THOUGHT: Failure flag set and/or leak rate {leak_rate} kg/hr exceeds 8 kg/hr threshold.\n"
            f"  Surge pressure of {surge_bar} bar detected — instantaneous closure risks fluid hammer.\n"
            f"RISK LEVEL: CRITICAL\n"
            f"RECOMMENDED ACTION: Notify field operator to inspect {system_code}-V01; "
            f"if shutdown required, apply staged valve closure over ≥{calc_fluid_transient_surge(telemetry['torque'])[1]}s.\n"
            f"CAVEAT: This assessment is illustrative. A qualified engineer must validate before any intervention."
        )
    elif leak_rate > 0.0:
        return (
            f"THOUGHT: Torque load {telemetry['torque']} Nm is above threshold; emission proxy elevated.\n"
            f"RISK LEVEL: WARNING\n"
            f"RECOMMENDED ACTION: Schedule preventative seal inspection on {system_code} during next maintenance window.\n"
            f"CAVEAT: Emission estimate is a proxy based on torque — direct measurement required for confirmation."
        )
    else:
        return (
            f"THOUGHT: All telemetry within normal operating range.\n"
            f"RISK LEVEL: NOMINAL\n"
            f"RECOMMENDED ACTION: Continue standard monitoring. No immediate action required.\n"
            f"CAVEAT: Automated assessments do not replace scheduled physical inspections."
        )


# ── VISUALISATION HELPERS ─────────────────────────────────────────────────────

def render_pid_schematic(file_path: str, system_code: str):
    """
    Renders a real P&ID PDF page if available, otherwise draws a
    clearly-labelled synthetic schematic.
    """
    if os.path.exists(file_path) and not os.path.isdir(file_path):
        try:
            import fitz
            doc  = fitz.open(file_path)
            page = doc.load_page(0)
            pix  = page.get_pixmap(matrix=fitz.Matrix(2, 2))
            return Image.open(io.BytesIO(pix.tobytes("png")))
        except Exception:
            pass

    # Synthetic fallback — clearly labelled as illustrative
    fig, ax = plt.subplots(figsize=(6.5, 3.2), facecolor="#0a192f")
    ax.set_facecolor("#0a192f")
    for i in range(11):
        ax.axhline(i / 10, color="#172a45", lw=0.6, alpha=0.7)
        ax.axvline(i / 10, color="#172a45", lw=0.6, alpha=0.7)

    ax.plot([0.1, 0.4, 0.4, 0.9], [0.5, 0.5, 0.7, 0.7],
            color="#64ffda", lw=4.5, label=f"{system_code} Process Line")
    ax.plot([0.4, 0.4], [0.62, 0.78], color="#f43f5e", lw=2.5)
    ax.fill([0.36, 0.44, 0.36, 0.44], [0.65, 0.75, 0.75, 0.65],
            color="#f43f5e", alpha=0.6)
    ax.text(0.4, 0.83, f"ACTUATOR VALVE\n{system_code}-V01",
            color="#f43f5e", ha="center", fontsize=8, fontweight="bold")
    ax.scatter([0.22], [0.5], color="#38bdf8", s=180, zorder=5,
               edgecolor="white", lw=1)
    ax.plot([0.22, 0.22], [0.5, 0.38], color="#38bdf8", lw=1.5, linestyle="--")
    ax.text(0.22, 0.30, f"TRANSMITTER\nPT-{system_code}-01",
            color="#38bdf8", ha="center", fontsize=8, fontweight="bold")

    ax.set_title(f"ILLUSTRATIVE SCHEMATIC: {system_code} MANIFOLD\n"
                 f"(Synthetic — not an engineering document)",
                 color="#e2e8f0", fontsize=9, fontweight="bold", pad=10)
    ax.text(0.98, 0.03,
            "Source: Equinor Huldra Open Data (public) — illustrative layout only",
            color="#475569", ha="right", fontsize=6, style="italic")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    plt.tight_layout()

    buf = io.BytesIO()
    plt.savefig(buf, format="png", facecolor=fig.get_facecolor(),
                edgecolor="none", dpi=150)
    plt.close(fig); buf.seek(0)
    return Image.open(buf)


def generate_architecture_diagram():
    """Static block diagram of the agent architecture."""
    fig, ax = plt.subplots(figsize=(6.5, 2.0), facecolor="#1e1e1e")
    ax.set_facecolor("#1e1e1e")
    box = dict(boxstyle="round,pad=0.4", facecolor="#114b4e",
               edgecolor="#00b4d8", lw=1.5)
    core = dict(boxstyle="round,pad=0.5", facecolor="#2d3748",
                edgecolor="#e2e8f0", lw=2)
    t = dict(color="white", ha="center", va="center",
             fontsize=7.5, fontweight="bold")

    ax.text(0.15, 0.75, "UCI Telemetry\n(Predictive Maint.)", bbox=box, **t)
    ax.text(0.15, 0.25, "ESA TROPOMI\n(Sentinel-5P CH4)",   bbox=box, **t)
    ax.text(0.50, 0.50, "Azure AI Foundry\nGPT-4o Agent",   bbox=core,
            color="white", ha="center", va="center", fontsize=8.5, fontweight="bold")
    ax.text(0.85, 0.75, "Equinor P&IDs\n(Open Data)",       bbox=box, **t)
    ax.text(0.85, 0.25, "MS Teams\nAdaptive Card",          bbox=box, **t)

    arr = dict(arrowstyle="->", color="#00b4d8", lw=1.5, mutation_scale=10)
    ax.annotate("", xy=(0.34, 0.53), xytext=(0.28, 0.70), arrowprops=arr)
    ax.annotate("", xy=(0.34, 0.47), xytext=(0.28, 0.30), arrowprops=arr)
    ax.annotate("", xy=(0.72, 0.70), xytext=(0.66, 0.53), arrowprops=arr)
    ax.annotate("", xy=(0.72, 0.30), xytext=(0.66, 0.47), arrowprops=arr)

    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    plt.tight_layout()
    return fig


def generate_emissions_plot(torque: float):
    """Torque vs estimated emission proxy curve with current operating point."""
    data = {
        "Torque_Nm":              [20,30,40,50,60,70,80,90,100,110,120],
        "Methane_Leak_Rate_kg_hr":[0,0,3,4.5,6,7.5,9,10.5,12,13.5,15]
    }
    df = pd.DataFrame(data)
    fig, ax = plt.subplots(figsize=(6, 3.0))
    ax.plot(df["Torque_Nm"], df["Methane_Leak_Rate_kg_hr"],
            color="#d9534f", marker="o", linewidth=2,
            label="Emission proxy curve (illustrative)")
    leak = max(0.0, (torque * 0.15) - 3.0) if torque > 40 else 0.0
    ax.scatter([torque], [leak], color="black", s=120, zorder=5,
               label=f"Current state ({torque} Nm)")
    ax.set_title("Torque Load vs. Fugitive Emission Proxy\n"
                 "(Illustrative — not a validated emissions model)",
                 fontsize=9, fontweight="bold")
    ax.set_xlabel("Torque (Nm)", fontsize=8)
    ax.set_ylabel("Emission proxy (kg/hr CH4 equiv.)", fontsize=8)
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend(fontsize=7)
    plt.tight_layout()
    return fig


# ── MAIN ENGINE ───────────────────────────────────────────────────────────────

def run_assessment(row_id: int, target_sensor: str):
    """Orchestrates all data sources and returns results for the Gradio UI."""
    system_code = target_sensor.split("-")[1]

    # 1. Telemetry
    tel       = fetch_uci_telemetry(row_id)
    power_kw  = (2 * math.pi * tel["rpm"] * tel["torque"]) / 60000
    power_dev = round(power_kw - 8.0, 2)
    leak_rate = round(max(0.0, (tel["torque"] * 0.15) - 3.0), 2) \
                if power_dev > 0 else 0.0
    surge_bar, safe_time = calc_fluid_transient_surge(tel["torque"])

    if tel["process_failure"] == 1 or leak_rate > 8.0:
        status = "🛑 CRITICAL — Field operator verification required"
    elif leak_rate > 0.0:
        status = "⚠️  WARNING — Elevated emission proxy detected"
    else:
        status = "✅ NOMINAL — Within safe operating limits"

    tel_summary = (
        f"Source: {tel['source']}\n"
        f"Failure flag : {'SET' if tel['process_failure'] == 1 else 'CLEAR'}\n"
        f"RPM          : {tel['rpm']}\n"
        f"Torque       : {tel['torque']} Nm\n"
        f"Tool wear    : {tel['tool_wear']} min\n"
        f"Power        : {round(power_kw,2)} kW  (dev: {power_dev:+} kW)"
    )

    # 2. Satellite
    sat = fetch_satellite_methane(
        bbox=[0.0, 59.0, 5.0, 62.0],
        date_range="2026-04-01/2026-06-09"
    )
    sat_summary = (
        f"Status    : {sat['status']}\n"
        f"Source    : {sat.get('source', 'N/A')}\n"
        f"Product   : {sat.get('product_type', 'N/A')}\n"
        f"Scene     : {sat['scene_id']}\n"
        f"Acquired  : {sat.get('acquired', 'N/A')}\n"
        f"Published : {sat.get('published', 'N/A')} "
        f"(origin: {sat.get('origin', 'N/A')})\n"
        f"Online    : {sat.get('online', 'N/A')}  |  "
        f"Size: {sat.get('size_mb', 0)} MB\n"
        f"CH4 note  : {sat.get('ch4_note', 'N/A')}"
    )

    # 3. Engineering documents (Equinor open data)
    extracted_dir = "/tmp/huldra_extracted_pids/HuldraPIDs"
    if os.path.exists(extracted_dir):
        files = os.listdir(extracted_dir)
        doc   = next((f for f in files
                      if system_code in f and f.upper().endswith(".PDF")),
                     "Generic_Safety_Manual.pdf")
        full_path = os.path.join(extracted_dir, doc)
        refs = count_pdf_references(full_path, system_code)
    else:
        doc       = f"C025-V-{system_code}-P-_E-001-01.PDF"
        full_path = doc
        refs      = 14

    doc_summary = (
        f"Source      : Equinor Huldra Open Data (public)\n"
        f"File        : {doc}\n"
        f"System code : {system_code}\n"
        f"References  : {refs} matches\n"
        f"Note        : In production, replace with client document store\n"
        f"              via Azure AI Search."
    )

    # 4. AI reasoning
    reasoning = get_foundry_assessment(tel, leak_rate, surge_bar, system_code)

    # 5. Teams Adaptive Card
    card = {
        "$schema": "http://adaptivecards.io/schemas/adaptive-card.json",
        "type":    "AdaptiveCard",
        "version": "1.4",
        "body": [
            {
                "type":   "TextBlock",
                "text":   f"Project Sentinel Alert — Asset {system_code}",
                "weight": "Bolder",
                "size":   "Medium",
                "color":  "Attention" if "CRITICAL" in status else "Warning"
            },
            {
                "type": "FactSet",
                "facts": [
                    {"title": "Sensor:",        "value": target_sensor},
                    {"title": "Status:",        "value": status},
                    {"title": "Emission proxy:","value": f"{leak_rate} kg/hr (illustrative)"},
                    {"title": "Surge risk:",    "value": f"{surge_bar} bar"},
                    {"title": "Drawing ref:",   "value": doc}
                ]
            },
            {
                "type": "TextBlock",
                "text": (
                    f"Recommended: Field operator to inspect {system_code}-V01.\n"
                    f"If valve closure required, apply staged sequence over "
                    f"≥{safe_time}s (Joukowski criterion).\n"
                    f"⚠ This alert is advisory only. Verify with qualified engineer."
                ),
                "wrap": True
            }
        ]
    }

    return (
        status,
        reasoning,
        tel_summary,
        sat_summary,
        doc_summary,
        render_pid_schematic(full_path, system_code),
        generate_emissions_plot(tel["torque"]),
        json.dumps(card, indent=2, ensure_ascii=False)
    )


# ── GRADIO UI ─────────────────────────────────────────────────────────────────

with gr.Blocks(theme=gr.themes.Default(primary_hue="teal",
                                        secondary_hue="slate")) as app:
    gr.Markdown("""
    # 🛡️ Project Sentinel — Industrial Asset Safety Agent
    ### Microsoft Agents League Hackathon 2026
    **Data sources:** UCI AI4I 2020 (CC BY 4.0) · ESA Sentinel-5P via Microsoft Planetary Computer · Equinor Huldra Open Data
    **AI engine:** Azure AI Foundry (GPT-4o) · **Alert routing:** Microsoft Teams Adaptive Cards
    > ⚠️ Prototype using public data only. Emission estimates are illustrative proxies,
    > not validated engineering outputs. Not for operational use.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### Controls")
            row_sel    = gr.Slider(1, 9999, value=164, step=1,
                                   label="UCI Dataset Row")
            sensor_sel = gr.Dropdown(
                choices=["PT-HB20-01", "PT-HA24-01", "PT-HO45-01"],
                value="PT-HB20-01",
                label="Asset Sensor Tag"
            )
            run_btn = gr.Button("Run Assessment", variant="primary")

        with gr.Column(scale=2):
            gr.Markdown("### Agent Architecture")
            gr.Plot(value=generate_architecture_diagram())

    status_box = gr.Textbox(label="System Status", lines=1)

    gr.Markdown("### Agent Reasoning (Azure AI Foundry)")
    reasoning_box = gr.Textbox(label="ReAct Trace", lines=8)

    gr.Markdown("### Data Sources")
    with gr.Row():
        tel_box = gr.Textbox(label="UCI Telemetry",    lines=6)
        sat_box = gr.Textbox(label="ESA Satellite CH4",lines=6)
    doc_box = gr.Textbox(label="Equinor P&ID Lookup", lines=5)

    gr.Markdown("### Asset Schematic")
    pid_img = gr.Image(label="Engineering Drawing", type="pil")

    gr.Markdown("### Emissions Profile")
    plot_box = gr.Plot(label="Torque vs Emission Proxy")

    gr.Markdown("### Teams Alert Payload")
    card_box = gr.Code(label="Adaptive Card JSON", language="json", lines=10)

    run_btn.click(
        fn=run_assessment,
        inputs=[row_sel, sensor_sel],
        outputs=[status_box, reasoning_box, tel_box, sat_box,
                 doc_box, pid_img, plot_box, card_box]
    )

app.launch(inline=True, share=True)